In [ ]:
%pip install -q torch torchvision pandas scikit-learn pillow matplotlib kagglehub


In [1]:
from pathlib import Path
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_DIR / "data" / "fashion-product-images-small"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "masterCategory"
MAX_IMAGES_PER_CLASS = 500
IMAGE_SIZE = 224
BATCH_SIZE = 32
BASELINE_EPOCHS = 5
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)
print("Device:", DEVICE)


Project directory: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai
Dataset directory: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai\data\fashion-product-images-small
Device: cpu


In [6]:
df = pd.read_csv(CSV_PATH, on_bad_lines="skip")

df.columns = df.columns.astype(str).str.strip()

print("CSV path:", CSV_PATH)
print("Columns:", df.columns.tolist())

LABEL_COLUMN = "masterCategory"

assert LABEL_COLUMN in df.columns

print("Dataset loaded correctly!")

CSV path: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai\data\fashion-product-images-small\styles.csv
Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']
Dataset loaded correctly!


In [7]:
df["image_path"] = df["id"].astype(str).map(
    lambda x: IMAGE_DIR / f"{x}.jpg"
)

df = df[df["image_path"].map(lambda x: x.is_file())].copy()

df = df.dropna(subset=[LABEL_COLUMN])

class_names = sorted(df[LABEL_COLUMN].unique().tolist())

class_to_idx = {
    name: index
    for index, name in enumerate(class_names)
}

df["label"] = df[LABEL_COLUMN].map(class_to_idx).astype(int)

print("Images:", len(df))
print("Classes:", class_names)

display(df[LABEL_COLUMN].value_counts())

Images: 44419
Classes: ['Accessories', 'Apparel', 'Footwear', 'Free Items', 'Home', 'Personal Care', 'Sporting Goods']


masterCategory
Apparel           21392
Accessories       11274
Footwear           9219
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64

In [9]:

class_counts = df["label"].value_counts()

df = df[
    df["label"].isin(class_counts[class_counts >= 5].index)
].copy()

class_names = sorted(df[LABEL_COLUMN].unique().tolist())

class_to_idx = {
    name: index
    for index, name in enumerate(class_names)
}

df["label"] = df[LABEL_COLUMN].map(class_to_idx).astype(int)

train_df, temporary_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))
print("Classes:", len(class_names))


Training: 31092
Validation: 6663
Testing: 6663
Classes: 6


In [12]:
class CustomCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

baseline_model = CustomCNN(len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)


In [13]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.detach().cpu().numpy())
        predicted_labels.extend(predictions.detach().cpu().numpy())

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(true_labels, predicted_labels)

    return average_loss, accuracy, true_labels, predicted_labels

baseline_history = []

for epoch in range(BASELINE_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        baseline_model, train_loader, baseline_optimizer
    )
    validation_loss, validation_accuracy, _, _ = run_epoch(
        baseline_model, validation_loader
    )

    baseline_history.append({
        "epoch": epoch + 1,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "train_loss": train_loss,
        "validation_loss": validation_loss
    })

    print(
        f"Baseline CNN Epoch {epoch + 1}/{BASELINE_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )

baseline_test_loss, baseline_accuracy, baseline_true, baseline_pred = run_epoch(
    baseline_model, test_loader
)

print("Custom CNN test accuracy:", baseline_accuracy)
print(classification_report(
    baseline_true,
    baseline_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))


NameError: name 'train_loader' is not defined

In [ ]:
weights = ResNet50_Weights.DEFAULT
transfer_model = models.resnet50(weights=weights)

for parameter in transfer_model.parameters():
    parameter.requires_grad = False

transfer_model.fc = nn.Linear(
    transfer_model.fc.in_features,
    len(class_names)
)

transfer_model = transfer_model.to(DEVICE)

head_optimizer = torch.optim.Adam(
    transfer_model.fc.parameters(),
    lr=0.001
)

head_history = []

for epoch in range(HEAD_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        transfer_model, train_loader, head_optimizer
    )
    validation_loss, validation_accuracy, _, _ = run_epoch(
        transfer_model, validation_loader
    )

    head_history.append({
        "stage": "new_classifier_head",
        "epoch": epoch + 1,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "train_loss": train_loss,
        "validation_loss": validation_loss
    })

    print(
        f"ResNet50 Head Epoch {epoch + 1}/{HEAD_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )


In [ ]:
for name, parameter in transfer_model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        parameter.requires_grad = True

fine_tune_optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, transfer_model.parameters()),
    lr=0.0001
)

fine_tune_history = []
best_validation_accuracy = 0.0
best_model_state = None

for epoch in range(FINE_TUNE_EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(
        transfer_model, train_loader, fine_tune_optimizer
    )
    validation_loss, validation_accuracy, _, _ = run_epoch(
        transfer_model, validation_loader
    )

    fine_tune_history.append({
        "stage": "fine_tuning",
        "epoch": epoch + 1,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "train_loss": train_loss,
        "validation_loss": validation_loss
    })

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in transfer_model.state_dict().items()
        }

    print(
        f"Fine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f}"
    )

if best_model_state is not None:
    transfer_model.load_state_dict(best_model_state)


In [ ]:
transfer_test_loss, transfer_accuracy, transfer_true, transfer_pred = run_epoch(
    transfer_model, test_loader
)

print("ResNet50 transfer learning test accuracy:", transfer_accuracy)
print(classification_report(
    transfer_true,
    transfer_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

comparison_df = pd.DataFrame([
    {"Model": "Custom CNN baseline", "Test Accuracy": baseline_accuracy},
    {"Model": "ResNet50 transfer learning + fine-tuning", "Test Accuracy": transfer_accuracy}
])

display(comparison_df.sort_values("Test Accuracy", ascending=False).reset_index(drop=True))

comparison_df.set_index("Model")["Test Accuracy"].plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    title="Custom CNN vs ResNet50 Transfer Learning"
)
plt.ylabel("Test Accuracy")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


In [ ]:
MODEL_PATH = MODEL_DIR / "day7_resnet50_transfer_learning.pth"
BASELINE_PATH = MODEL_DIR / "day7_custom_cnn_baseline.pth"
COMPARISON_PATH = REPORT_DIR / "day7_model_comparison.csv"
HISTORY_PATH = REPORT_DIR / "day7_training_history.csv"
CLASS_NAMES_PATH = MODEL_DIR / "day7_class_names.json"

torch.save({
    "model_state_dict": transfer_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "architecture": "ResNet50"
}, MODEL_PATH)

torch.save({
    "model_state_dict": baseline_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "architecture": "CustomCNN"
}, BASELINE_PATH)

history_df = pd.DataFrame(
    baseline_history
    + head_history
    + fine_tune_history
)

comparison_df.to_csv(COMPARISON_PATH, index=False)
history_df.to_csv(HISTORY_PATH, index=False)

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=4)

print("Saved ResNet50 model:", MODEL_PATH)
print("Saved baseline CNN:", BASELINE_PATH)
print("Saved comparison:", COMPARISON_PATH)
print("Saved training history:", HISTORY_PATH)
print("Saved class names:", CLASS_NAMES_PATH)
